# ESCM 2.0 — Work package 3: biologically inspired spatial interactions

## Research question

Can **short-range cooperative excitation with broader lateral inhibition** maintain a multi-pixel edge without retaining obsolete edge positions? The original ESCM preserves individual pixels effectively but often extinguishes a simultaneous edge or selects only one surviving pixel (notebooks 08–09).

This experiment investigates a well-established **neural-field motif** rather than introducing a custom recency rule. Amari (1977) studied spatially structured excitatory/inhibitory interactions and persistent activity patterns. The application here is deliberately phenomenological: one ESCM unit still delivers effective lateral excitation and inhibition, rather than representing separate biological excitatory and inhibitory cell types.

**References:** Amari, *Biological Cybernetics* 27, 77–87 (1977), DOI 10.1007/BF00337259; Kishimoto & Amari, *Journal of Mathematical Biology* 7, 303–318 (1979), DOI 10.1007/BF00275151.

**Controlled change:** preserve the same Brian2 LIF equations, input assignment, self-feedback, persistence gate, integration method and 0.5 ms clock as the **counter-free ESCM**. Change *only the spatial lateral weights*.

The experiment has four safeguards:

1. Validate the weight geometry, signs and absence of unintended self-connections.
2. Compare with both the uniform-inhibition baseline and component ablations.
3. Select parameters on **training stimuli**, then evaluate on different **held-out stimuli**.
4. Evaluate spatial fidelity, obsolete activity, off-target activity, spike timing and timestep sensitivity—not just total spike counts.

**Scope:** This is a small (11 × 9) scientific screening experiment. It does not establish biological realism, full-camera scalability, or chip-level performance.

## 1. Mathematical model and stimuli

The membrane and synaptic dynamics remain

\[
\tau\dot v_i=-v_i+\mathrm{ExtIn}_i+P_i(\mathrm{Exc}_i+\mathrm{Inh}_i),\qquad
\tau\dot z_i=-z_i,\quad z\in\{\mathrm{ExtIn},\mathrm{Exc},\mathrm{Inh}\}.
\]

A camera event **sets** \(\mathrm{ExtIn}_i=\beta\). A neuronal spike **sets** its own self-excitatory trace to \(W_{\rm self}\). Lateral excitation **adds** positive contributions, while lateral inhibition **adds** negative contributions multiplied by postsynaptic \(P\).

For presynaptic \(j\) and postsynaptic \(i\), the proposed spatial kernels are

\[
w^E_{ij}=A_E e^{-d_{ij}^2/(2\sigma_E^2)},\qquad
w^I_{ij}=A_I e^{-d_{ij}^2/(2\sigma_I^2)},\quad \sigma_E<\sigma_I,
\]

with \(d_{ij}\) in **pixels**, nonself connections within a Chebyshev radius of four, and positive amplitudes \(A_E,A_I\). Inhibition enters the postsynaptic variable with a minus sign.

We only screen kernels with **net positive interaction at one pixel** and **net negative interaction at three pixels**. This is a necessary spatial-shape condition for the proposed local-excitation/surround-inhibition motif; it is *not* sufficient for stable persistence.

**Training stimuli:** simultaneous moving five-pixel edge, staggered moving edge, stationary five-pixel edge, two stationary separated edges.

**Held-out stimuli:** a single pixel, a fast moving edge, a slow moving edge, and a moving edge with one isolated noise event. All event times are multiples of 0.5 ms.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import product
from tqdm.auto import tqdm
from brian2 import *

prefs.codegen.target = "numpy"

width, height, radius = 11, 9, 4
N = width * height
edge_rows = np.arange(2, 7)
edge_columns = np.arange(3, 8)
final_edge = edge_rows * width + edge_columns[-1]
previous_edges = np.concatenate([edge_rows * width + x for x in edge_columns[:-1]])
center = 4 * width + 5

default_dt_ms = 0.5
end_ms = 70.0
evaluation_start_ms = 50.0
tau_model = 0.2 * ms
threshold_model, reset_model = 0.1, 0.0
input_weight, self_weight = 0.5, 6.0

def edge_input(interval, stagger=0.0):
    ids, times = [], []
    for step, x in enumerate(edge_columns):
        for row_index, y in enumerate(edge_rows):
            ids.append(y * width + x)
            times.append(1.0 + interval * step + stagger * row_index)
    return np.array(ids, dtype=int), np.array(times)

training = {
    "moving_simultaneous": edge_input(4.0, 0.0),
    "moving_staggered": edge_input(4.0, 0.5),
    "stationary_edge": (edge_rows*width + 5, np.ones(5)),
    "two_stationary_edges": (
        np.r_[edge_rows*width+2, edge_rows*width+8],
        np.ones(10),
    ),
}
held_out = {
    "single_pixel": (np.array([center]), np.array([1.0])),
    "fast_edge": edge_input(2.0, 0.5),
    "slow_edge": edge_input(8.0, 0.5),
    "noisy_edge": (
        np.r_[edge_input(4.0, 0.5)[0], 0],
        np.r_[edge_input(4.0, 0.5)[1], 6.0],
    ),
}
def desired_pixels(name):
    if name in ("moving_simultaneous", "moving_staggered",
                "fast_edge", "slow_edge", "noisy_edge"):
        return final_edge
    if name == "stationary_edge":
        return edge_rows*width + 5
    if name == "two_stationary_edges":
        return np.r_[edge_rows*width+2, edge_rows*width+8]
    return np.array([center])

for name, (ids, times) in {**training, **held_out}.items():
    assert times.max() < evaluation_start_ms
    assert np.allclose(times / default_dt_ms, np.rint(times / default_dt_ms))
    assert len(set(zip((times/default_dt_ms).astype(int), ids))) == len(ids)
print("Training protocols:", list(training))
print("Held-out protocols:", list(held_out))

## 2. Verify the spatial kernel before simulating

A kernel is a scientific assumption, not just an implementation detail. Check all directed edges and their weights explicitly:

- no nonself lateral connection has \(i=j\);
- every edge satisfies \(|\Delta x|\le4\), \(|\Delta y|\le4\);
- weights are nonnegative before the inhibitory sign;
- spatial weights are symmetric for \(i\leftrightarrow j\);
- the nearest interaction is net excitatory and the three-pixel interaction net inhibitory;
- the inhibitory kernel is spatially broader than the excitatory kernel.

These checks verify **geometry and the intended kernel shape**, not network stability. We display the effective net kernel (excitation minus inhibition) for a representative configuration.

In [ ]:
source = np.repeat(np.arange(N), N)
target = np.tile(np.arange(N), N)
dx = source % width - target % width
dy = source // width - target // width
within = (source != target) & (np.abs(dx) <= radius) & (np.abs(dy) <= radius)
src, dst = source[within], target[within]
dist2 = (dx[within]**2 + dy[within]**2).astype(float)
assert np.all(src != dst)
assert np.all(abs(src % width - dst % width) <= radius)
assert np.all(abs(src // width - dst // width) <= radius)
assert len(set(zip(src, dst))) == len(src)
assert set(zip(src, dst)) == set(zip(dst, src))
print("Lateral directed connections:", len(src))
print("Directed self-connections (must be zero):", np.count_nonzero(src == dst))

sigma_I = 2.5  # fixed here; test alternative scales after this first screening
candidates = []
for amp_E, amp_I, sigma_E in product((1.5, 3.0, 5.0),
                                     (0.5, 1.5, 3.0),
                                     (0.8, 1.25)):
    w_near = amp_E*np.exp(-1/(2*sigma_E**2)) - amp_I*np.exp(-1/(2*sigma_I**2))
    w_far = amp_E*np.exp(-9/(2*sigma_E**2)) - amp_I*np.exp(-9/(2*sigma_I**2))
    if w_near > 0 and w_far < 0:
        candidates.append((amp_E, amp_I, sigma_E))
print(f"Candidate kernels satisfying spatial-shape constraints: {len(candidates)}")

example = candidates[0]
amp_E, amp_I, sigma_E = example
offset = np.arange(-radius, radius+1)
xx, yy = np.meshgrid(offset, offset)
rr = xx**2 + yy**2
net_kernel = amp_E*np.exp(-rr/(2*sigma_E**2)) - amp_I*np.exp(-rr/(2*sigma_I**2))
net_kernel[radius, radius] = self_weight  # explicitly separate autapse
fig, ax = plt.subplots(figsize=(6, 5))
limit = max(abs(net_kernel.min()), abs(net_kernel.max()))
im = ax.imshow(net_kernel, origin="lower", cmap="coolwarm", vmin=-limit, vmax=limit)
ax.set(xlabel="Horizontal offset (pixels)", ylabel="Vertical offset (pixels)",
       title=f"Net lateral kernel (E={amp_E}, I={amp_I}, sigmaE={sigma_E}, sigmaI={sigma_I})")
ax.set_xticks(np.arange(len(offset)), labels=offset)
ax.set_yticks(np.arange(len(offset)), labels=offset)
fig.colorbar(im, ax=ax, label="Net weight (model units)")
plt.show()

## 3. Implement the models in Brian2

The **uniform baseline** is the tested counter-free ESCM rule: no lateral excitation, and a weight of 3 for every lateral neighbor. In the **spatial candidate**, the same grid receives separate positive and negative Gaussian weights.

To make the conceptual autapse explicit, we use a dedicated **one-to-one self-synapse** instead of repeatedly setting \(\mathrm{Exc}_{\rm pre}\) through every outgoing lateral synapse. The self-synapse still executes on the neuron's spike and **assigns** \(\mathrm{Exc}=6\), matching the original operation's intent.

The order is explicit: external input (−2), self-refresh (−1), lateral excitation (0), lateral inhibition (1). This prevents dependence on object creation order. The persistence gate is unchanged.

The spatial weights represent **effective circuit interactions**; a more anatomical implementation may later split the inhibitory effect into an interneuron population.

In [ ]:
def simulate(mode, ids, times_ms, amp_E=0.0, amp_I=3.0,
             sigma_E=1.0, step_ms=default_dt_ms, keep_states=False):
    start_scope()
    defaultclock.dt = step_ms * ms
    assert np.allclose(times_ms/step_ms, np.rint(times_ms/step_ms))

    constants = {
        "tau_model": tau_model, "threshold_model": threshold_model,
        "reset_model": reset_model, "input_weight": input_weight,
        "self_weight": self_weight,
    }
    camera = SpikeGeneratorGroup(N, ids, times_ms * ms)
    eqs = """
    dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau_model : 1
    dExtIn/dt = -ExtIn/tau_model : 1
    dExc/dt = -Exc/tau_model : 1
    dInh/dt = -Inh/tau_model : 1
    P : integer
    """
    neurons = NeuronGroup(
        N, eqs, threshold="v > threshold_model",
        reset="v = reset_model", refractory=0*ms,
        events={"P_ON": "v > threshold_model",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= threshold_model)"},
        method="exact", namespace=constants,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")

    feed = Synapses(camera, neurons, on_pre="ExtIn_post = input_weight",
                    namespace=constants)
    feed.connect(j="i")
    feed.pre.order = -2

    self_syn = Synapses(neurons, neurons,
                        on_pre="Exc_post = self_weight", namespace=constants)
    self_syn.connect(j="i")
    self_syn.pre.order = -1

    lateral = Synapses(
        neurons, neurons, model="w_e : 1 (constant)\nw_i : 1 (constant)",
        on_pre={"pre": "Exc_post += w_e",
                "pre_2": "Inh_post -= P_post * w_i"},
    )
    lateral.connect(i=src, j=dst)
    lateral.pre.order = 0
    lateral.pre_2.order = 1
    if mode == "uniform":
        lateral.w_e = 0.0
        lateral.w_i = 3.0
    elif mode == "spatial":
        lateral.w_e = amp_E * np.exp(-dist2/(2*sigma_E**2))
        lateral.w_i = amp_I * np.exp(-dist2/(2*sigma_I**2))
    else:
        raise ValueError(mode)

    output = SpikeMonitor(neurons)
    if keep_states:
        states = StateMonitor(neurons, ["v", "P", "Exc", "Inh"],
                              record=[center], dt=defaultclock.dt)
        network = Network(camera, neurons, feed, self_syn, lateral, output, states)
    else:
        network = Network(camera, neurons, feed, self_syn, lateral, output)

    network.run(end_ms*ms)
    result = {
        "t": np.asarray(output.t/ms),
        "i": np.asarray(output.i, dtype=int),
    }
    if keep_states:
        result["state_time"] = np.asarray(states.t/ms)
        result["state"] = {key: np.asarray(getattr(states, key)[0]).copy()
                           for key in ("v", "P", "Exc", "Inh")}
    return result

### Check that the explicit autapse preserves the baseline

Before evaluating any new lateral weights, compare the **uniform-inhibition baseline using a dedicated one-to-one self-synapse** against the established counter-free implementation from notebook 09, which refreshed \(\mathrm{Exc}_{\rm pre}\) through every outgoing lateral connection.

This assertion compares every output spike's timestep and neuron index on **all four training protocols**. If it fails, stop: the refactoring of the autapse has changed baseline behavior, and a comparison of spatial kernels would be confounded.

In [ ]:
def legacy_counterfree(ids, times_ms):
    """Reference from notebook 09: self-refresh through outgoing synapses."""
    start_scope()
    defaultclock.dt = default_dt_ms * ms
    params = dict(tau_model=tau_model, threshold_model=threshold_model,
                  reset_model=reset_model, input_weight=input_weight,
                  self_weight=self_weight, inhibitory_weight=3.0)
    camera = SpikeGeneratorGroup(N, ids, times_ms*ms)
    eqs = """
    dv/dt = (-v + ExtIn + P*(Exc + Inh))/tau_model : 1
    dExtIn/dt = -ExtIn/tau_model : 1
    dExc/dt = -Exc/tau_model : 1
    dInh/dt = -Inh/tau_model : 1
    P : integer
    """
    neurons = NeuronGroup(
        N, eqs, threshold="v > threshold_model",
        reset="v = reset_model", refractory=0*ms,
        events={"P_ON": "v > threshold_model",
                "P_OFF": "(timestep(t - lastspike, dt) > timestep(dt, dt) and v <= threshold_model)"},
        method="exact", namespace=params,
    )
    neurons.set_event_schedule("P_ON", when="after_thresholds")
    neurons.run_on_event("P_ON", "P = 1", when="after_thresholds")
    neurons.set_event_schedule("P_OFF", when="groups")
    neurons.run_on_event("P_OFF", "P = 0", when="groups")
    feed = Synapses(camera, neurons, on_pre="ExtIn_post = input_weight",
                    namespace=params)
    feed.connect(j="i")
    recurrent = Synapses(
        neurons, neurons,
        on_pre={"pre": "Exc_pre = self_weight",
                "pre_2": "Inh_post -= P_post * inhibitory_weight"},
        namespace=params,
    )
    recurrent.pre.order = 0
    recurrent.pre_2.order = 1
    recurrent.connect(i=src, j=dst)
    output = SpikeMonitor(neurons)
    Network(camera, neurons, feed, recurrent, output).run(end_ms*ms)
    ticks = np.rint(np.asarray(output.t/ms)/default_dt_ms).astype(int)
    return np.column_stack((ticks, np.asarray(output.i, dtype=int)))

for name, stimulus in tqdm(training.items(), desc="Baseline equivalence"):
    original = legacy_counterfree(*stimulus)
    refactored = simulate("uniform", *stimulus)
    ticks = np.rint(refactored["t"]/default_dt_ms).astype(int)
    newer = np.column_stack((ticks, refactored["i"]))
    if not np.array_equal(original, newer):
        n = min(len(original), len(newer))
        first = next((j for j in range(n)
                      if not np.array_equal(original[j], newer[j])), n)
        raise AssertionError(
            f"Baseline changed for {name}: old={len(original)}, new={len(newer)}, "
            f"first mismatch at output index {first}"
        )
    print(f"{name}: exact spike parity PASS ({len(original)} spikes)")

## 4. Evaluate spatial representation, including failures

A pixel is considered active if it generates at least one output spike in the **final 20 ms** (50–70 ms). This is an operational readout, not a claim that spikes equal continuous intensity.

For a desired set \(G\) and observed active set \(A\),

\[
R=|A\cap G|/|G|,\quad
Q=|A\cap G|/|A|,\quad
F_1=2RQ/(R+Q),
\]

with zero scores when no pixels are active. For moving edges, we also report how many previously stimulated locations remain active and how many **unstimulated** pixels fire.

During movement we additionally evaluate a **2 ms readout window following each column's onset**, providing a measure of tracking, not just the final state. This readout is imperfect for strongly staggered input, and must be interpreted accordingly.

**Selection rule:** maximize the mean training final-state F1, then mean moving-tracking F1. Held-out stimuli are evaluated only afterward. We also report raw counts so a perfect score on one tiny protocol is not mistaken for robust behavior.

In [ ]:
def metrics(result, desired, old, stimulated):
    active = np.unique(result["i"][(result["t"] >= evaluation_start_ms)
                                   & (result["t"] < end_ms)])
    hit = np.intersect1d(active, desired).size
    recall = hit / len(desired)
    precision = hit / len(active) if len(active) else 0.0
    f1 = 2*recall*precision/(recall+precision) if recall+precision else 0.0
    return dict(
        F1=f1, recall=recall, precision=precision,
        active=len(active),
        retained=int(hit),
        obsolete=int(np.isin(active, old).sum()),
        unprovoked=int((~np.isin(active, stimulated)).sum()),
        output_spikes=len(result["i"]),
    )

def assess(name, input_data, result):
    ids, times = input_data
    desired = desired_pixels(name)
    old = previous_edges if name in (
        "moving_simultaneous", "moving_staggered", "fast_edge",
        "slow_edge", "noisy_edge") else np.array([], dtype=int)
    values = metrics(result, desired, old, np.unique(ids))
    if name.startswith("moving_") or name in ("fast_edge", "slow_edge", "noisy_edge"):
        tracking = []
        for col in edge_columns:
            onset = times[np.isin(ids, edge_rows*width + col)].min()
            present = np.unique(result["i"][
                (result["t"] >= onset + 0.5) & (result["t"] < onset + 2.5)])
            target = edge_rows*width+col
            match = np.intersect1d(present, target).size
            prec = match/len(present) if len(present) else 0.0
            rec = match/len(target)
            tracking.append(2*prec*rec/(prec+rec) if prec+rec else 0)
        values["tracking_F1"] = float(np.mean(tracking))
    else:
        values["tracking_F1"] = np.nan
    return values

runs = []
for name, stimulus in tqdm(training.items(), desc="Uniform baseline"):
    r = simulate("uniform", *stimulus)
    runs.append(dict(model="uniform", kernel="uniform", protocol=name,
                     **assess(name, stimulus, r)))

for amp_E, amp_I, sigma_E in tqdm(candidates, desc="Structured kernels"):
    kernel = f"E{amp_E:g}_I{amp_I:g}_sE{sigma_E:g}"
    for name, stimulus in training.items():
        r = simulate("spatial", *stimulus, amp_E=amp_E,
                     amp_I=amp_I, sigma_E=sigma_E)
        runs.append(dict(model="spatial", kernel=kernel, protocol=name,
                         amp_E=amp_E, amp_I=amp_I, sigma_E=sigma_E,
                         **assess(name, stimulus, r)))

train_results = pd.DataFrame(runs)
print("Number of model × protocol runs:", len(train_results))
print("\nUniform baseline:")
display(train_results[train_results.model == "uniform"])
ranked = (train_results[train_results.model == "spatial"]
          .groupby("kernel", as_index=False)
          .agg(mean_F1=("F1", "mean"),
               mean_tracking_F1=("tracking_F1", "mean"),
               mean_obsolete=("obsolete", "mean"))
          .sort_values(["mean_F1", "mean_tracking_F1"], ascending=False))
print("\nTraining ranking:")
display(ranked)
winning_label = ranked.iloc[0]["kernel"]
winner = train_results[train_results.kernel == winning_label].iloc[0]
best_config = dict(amp_E=float(winner.amp_E),
                   amp_I=float(winner.amp_I),
                   sigma_E=float(winner.sigma_E))
print("Chosen spatial kernel (training only):", winning_label, best_config)

## 5. Mechanistic controls and held-out evaluation

A spatial candidate can appear successful for the wrong reason—for instance, if lateral inhibition becomes negligible and all historical pixels persist. Therefore we explicitly evaluate:

- **Uniform inhibition:** counter-free ESCM reference.
- **Spatial E+I:** candidate selected from the training protocols.
- **Spatial excitation only:** identical excitation, zero inhibition.
- **Spatial inhibition only:** identical inhibition, no lateral excitation.

These are **causal component ablations** around the selected candidate, not a new parameter search. The held-out set contains faster/slower edges, a single input, and a noisy edge. Report the scores for **every** protocol. A failure on held-out patterns is evidence against generality even if the training F1 is high.

In [ ]:
models = {
    "uniform": dict(mode="uniform"),
    "spatial_EI": dict(mode="spatial", **best_config),
    "excitation_only": dict(mode="spatial", **{**best_config, "amp_I": 0.0}),
    "inhibition_only": dict(mode="spatial", **{**best_config, "amp_E": 0.0}),
}
holdout_rows = []
holdout_spikes = {}
for model_name, config in tqdm(models.items(), desc="Held-out model"):
    for name, stimulus in held_out.items():
        r = simulate(**config, ids=stimulus[0], times_ms=stimulus[1])
        holdout_spikes[model_name, name] = r
        holdout_rows.append(dict(model=model_name, protocol=name,
                                 **assess(name, stimulus, r)))
holdout_results = pd.DataFrame(holdout_rows)
display(holdout_results.sort_values(["protocol", "model"]))

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
for ax, quantity, ylabel in zip(axes, ("F1", "tracking_F1"),
                                 ("Final-state F1", "Moving-edge tracking F1")):
    table = holdout_results.pivot(index="model", columns="protocol", values=quantity)
    table.plot(kind="bar", ax=ax)
    ax.set(ylabel=ylabel, xlabel="Model", ylim=(0, 1),
           title="Held-out performance")
    ax.tick_params(axis="x", rotation=20)
    ax.legend(title="Protocol", fontsize=8)
plt.show()

## 6. Verify whether the chosen mechanism depends on the simulation clock

The original model has \(\tau=0.2\) ms but uses a 0.5 ms clock. Numerical integration uses Brian2's exact linear state update, yet thresholding and event execution still occur discretely.

Repeat two representative protocols at **0.5 and 0.25 ms** with *the same physical event times*, and compare their **functional outputs**. Spike trains are not expected to be tick-identical across different timestep sizes.

We also visualize a representative moving-edge raster. The intended result is an active **population** at the newest edge, not merely a large output event count.

In [ ]:
clock_rows = []
for step_ms in (0.5, 0.25):
    for model_name in ("uniform", "spatial_EI"):
        config = models[model_name]
        for name in ("moving_staggered", "stationary_edge"):
            stimulus = training[name]
            r = simulate(**config, ids=stimulus[0], times_ms=stimulus[1],
                         step_ms=step_ms)
            clock_rows.append(dict(step_ms=step_ms, model=model_name,
                                   protocol=name, **assess(name, stimulus, r)))
display(pd.DataFrame(clock_rows))

chosen_input = training["moving_staggered"]
fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True,
                         constrained_layout=True)
for ax, model_name in zip(axes, ("uniform", "spatial_EI")):
    result = simulate(**models[model_name], ids=chosen_input[0],
                      times_ms=chosen_input[1])
    ax.scatter(result["t"], result["i"], s=12, label="Output")
    ax.scatter(chosen_input[1], chosen_input[0], marker="x",
               color="tab:red", s=50, label="Input")
    ax.axvspan(evaluation_start_ms, end_ms, alpha=0.08, color="gray")
    ax.set(title=model_name, ylabel="Neuron index", ylim=(0, N))
axes[0].legend()
axes[-1].set(xlabel="Time (ms)", xlim=(0, end_ms))
plt.show()

## 7. Interpret before changing ESCM 2.0

This notebook is designed to distinguish three possible outcomes:

1. **Structured E–I improves both retention and suppression:** investigate robustness to more kernels, different edge orientations, event rates and recorded DVS data.
2. **Structured E–I preserves the edge but leaves obsolete trails:** local cooperation exists, but inhibitory competition is insufficient. Do not claim success merely from high retention.
3. **Structured E–I collapses the edge or becomes highly timestep-sensitive:** the proposed motif is not sufficient under these neuronal dynamics; reconsider its interaction with \(P\), self-excitation and the strong discrete-time reset.

The current neuron combines effective excitation and inhibition in the same presynaptic population; this is a **neural-field approximation**, not a fully Dale-compliant cortical microcircuit. Likewise, passing a small synthetic test does not demonstrate event-camera usefulness, hardware compatibility, or biological equivalence.

**Next evidence needed:** Compare the winning candidate against the original on a fixed recording from Event-Human3.6m, and evaluate the trade-off between spatial coverage, persistence, obsolete output and computational cost. Only then consider short-term synaptic depression or explicit inhibitory interneurons.

**Reproducibility:** Save the executed notebook with Python, NumPy and Brian2 versions and the resulting tables. The parameter search is finite and hand-chosen, and conclusions apply only to these settings.